# 🪙 **Cryptocurrency Web Scraping & Live Market Analysis Dashboard**

An end-to-end data engineering and analytics application that scrapes real-time cryptocurrency metrics from the web, processes market dynamics using Pandas, and visualizes asset performance via an interactive Streamlit dashboard.

---

## 📌 Project Overview
This project extracts, transforms, and visualizes live market data for top cryptocurrencies to provide actionable market 
intelligence. It features automated scraping, real-time metric tracking, and dynamic interactive visual insights.

### Key Capabilities
* **Automated Web Scraping:** Uses `BeautifulSoup` and `Requests` to parse dynamic web elements and 
extract asset pricing, 24-hour performance percentage changes, and total market capitalization.

* **Aggregated Market Insights:** Computes real-time market statistics, identifying top market gainers and losers dynamically.


* **Interactive Data Visualizations:** Built with `Plotly` and `Streamlit` to deliver customizable charts and adaptable view parameters.

---

## 🛠️ Tech Stack & Dependencies
* **Programming Language:** Python
* **Data Extraction:** BeautifulSoup4, Requests
* **Data Manipulation:** Pandas
* **Data Visualization:** Plotly Express
* **Web Application:** Streamlit
* **Development Environment:** Visual Studio Code

## Step 1

---

This Python script builds an interactive Streamlit web dashboard that automatically scrapes real-time cryptocurrency data (names, symbols, prices, 24-hour percentage changes, and market capitalizations) from CoinMarketCap using `requests` and `BeautifulSoup`. To prevent rate-limiting and optimize performance, the data extraction function is cached for 5-minute intervals via `@st.cache_data`. Users can interactively adjust the number of tracked cryptocurrencies (5 to 50) or trigger manual data refreshes using sidebar controls. The main application interface cleans and formats the raw HTML data into a Pandas DataFrame, dynamic summary cards highlighting total assets alongside the top 24-hour gainer and loser, an interactive raw data table, and side-by-side dark-themed Plotly bar charts comparing market capitalizations and 24-hour price momentum.

In [25]:
# Import Python Libraries
import streamlit as st
import requests
from bs4 import BeautifulSoup
import pandas as pd
import plotly.express as px

In [26]:
# Page layout configuration
st.set_page_config(
    page_title="Crypto Market Tracker",
    page_icon="🪙",
    layout="wide"
)

def parse_numeric_value(value):
    cleaned = value.replace("$", "").replace(",", "").replace("%", "").strip()
    multipliers = {"K": 1_000, "M": 1_000_000, "B": 1_000_000_000, "T": 1_000_000_000_000}
    multiplier = multipliers.get(cleaned[-1].upper(), 1) if cleaned else 1
    number = cleaned[:-1] if cleaned and cleaned[-1].upper() in multipliers else cleaned
    return float(number) * multiplier

# Function to scrape live data from CoinMarketCap
@st.cache_data(ttl=300)  # This caches results for 5 minutes to avoid rate limits
def scrape_crypto_data(top_n=20):
    url = "https://coinmarketcap.com/"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }

    response = requests.get(url, headers=headers, timeout=30)
    if response.status_code != 200:
        st.error(f"Failed to fetch data. HTTP Status: {response.status_code}")
        return pd.DataFrame()

    soup = BeautifulSoup(response.text, "html.parser")
    table_rows = soup.select("tbody tr")[:top_n]

    crypto_list = []
    for row in table_rows:
        cols = row.find_all("td")
        if len(cols) < 8:
            continue

        try:
            name_p = cols[2].find_all("p")
            name = name_p[0].text if name_p else "N/A"
            symbol = name_p[1].text if len(name_p) > 1 else "N/A"

            price_text = cols[3].text
            change_24h_text = cols[5].text
            market_cap_text = cols[7].text

            crypto_list.append({
                    "Name": name,
                    "Symbol": symbol,
                    "Price ($)": parse_numeric_value(price_text),
                    "24h Change (%)": parse_numeric_value(change_24h_text),
                    "Market Cap ($)": parse_numeric_value(market_cap_text)
                })
        except (AttributeError, ValueError, IndexError):
            continue

    return pd.DataFrame(crypto_list)

2026-09-15 23:53:28.143 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 23:53:28.177 No runtime found, using MemoryCacheStorageManager


In [27]:
# Dashboard Header
st.title("🪙 Real-Time Crypto Market Dashboard")
st.markdown("Live market data, portfolio simulation, and risk signals in one place.")

@st.cache_data(ttl=300)
def fetch_fear_greed_index():
    """Fetch the current Crypto Fear & Greed Index."""
    try:
        response = requests.get(
            "https://api.alternative.me/fng/?limit=1",
            headers={"User-Agent": "crypto-market-dashboard/1.0"},
            timeout=15,
        )
        response.raise_for_status()
        payload = response.json()
        item = payload.get("data", [{}])[0]
        return item.get("value", "N/A"), item.get("value_classification", "Unavailable")
    except (requests.RequestException, ValueError, IndexError, AttributeError):
        return "N/A", "Unavailable"

# Sidebar controls
st.sidebar.header("Dashboard Controls")
num_coins = st.sidebar.slider("Number of Cryptocurrencies", min_value=5, max_value=50, value=15, step=5)

if st.sidebar.button("Refresh Data"):
    st.cache_data.clear()
    st.rerun()

st.sidebar.subheader("Paper Portfolio")
st.sidebar.caption("One holding per line: SYMBOL, quantity, average cost in USD")
portfolio_text = st.sidebar.text_area(
    "Holdings",
    value="BTC, 0.25, 60000\nETH, 2.5, 3000",
    height=100,
)

# Main Application Logic
df = scrape_crypto_data(top_n=num_coins)

if not df.empty:
    df["Symbol"] = df["Symbol"].str.upper().str.strip()
    top_gainer = df.loc[df["24h Change (%)"].idxmax()]
    top_loser = df.loc[df["24h Change (%)"].idxmin()]
    fear_greed_value, fear_greed_label = fetch_fear_greed_index()

    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Total Cryptos Tracked", len(df))
    col2.metric(
        "Top Gainer (24h)",
        f"{top_gainer['Name']} ({top_gainer['Symbol']})",
        f"{top_gainer['24h Change (%)']:.2f}%",
    )
    col3.metric(
        "Top Loser (24h)",
        f"{top_loser['Name']} ({top_loser['Symbol']})",
        f"{top_loser['24h Change (%)']:.2f}%",
    )
    col4.metric("Fear & Greed", fear_greed_label, fear_greed_value)

    portfolio_rows = []
    for line_number, line in enumerate(portfolio_text.splitlines(), start=1):
        if not line.strip():
            continue
        parts = [part.strip() for part in line.split(",")]
        if len(parts) != 3:
            st.sidebar.warning(f"Holding line {line_number} must have 3 comma-separated values.")
            continue
        symbol, quantity_text, cost_text = parts
        try:
            quantity = float(quantity_text)
            average_cost = float(cost_text)
        except ValueError:
            st.sidebar.warning(f"Holding line {line_number} contains an invalid number.")
            continue
        match = df[df["Symbol"] == symbol.upper()]
        if match.empty:
            st.sidebar.info(f"{symbol.upper()} is not in the scraped top {num_coins}.")
            continue
        current_price = match.iloc[0]["Price ($)"]
        market_value = quantity * current_price
        invested_value = quantity * average_cost
        portfolio_rows.append(
            {
                "Symbol": symbol.upper(),
                "Quantity": quantity,
                "Average Cost ($)": average_cost,
                "Current Price ($)": current_price,
                "Market Value ($)": market_value,
                "P/L ($)": market_value - invested_value,
                "P/L (%)": ((market_value / invested_value) - 1) * 100 if invested_value else 0,
            }
        )

    if portfolio_rows:
        portfolio_df = pd.DataFrame(portfolio_rows)
        total_value = portfolio_df["Market Value ($)"].sum()
        total_invested = (portfolio_df["Average Cost ($)"] * portfolio_df["Quantity"]).sum()
        total_pl = portfolio_df["P/L ($)"].sum()
        portfolio_col1, portfolio_col2, portfolio_col3 = st.columns(3)
        portfolio_col1.metric("Portfolio Value", f"${total_value:,.2f}")
        portfolio_col2.metric("Portfolio P/L", f"${total_pl:,.2f}", f"{(total_pl / total_invested * 100) if total_invested else 0:.2f}%")
        portfolio_col3.metric("Invested Capital", f"${total_invested:,.2f}")

        allocation_col, table_col = st.columns([1, 2])
        with allocation_col:
            allocation_fig = px.pie(
                portfolio_df,
                names="Symbol",
                values="Market Value ($)",
                hole=0.55,
                title="Portfolio Allocation",
                template="plotly_dark",
            )
            st.plotly_chart(allocation_fig, use_container_width=True)
        with table_col:
            st.subheader("Paper Portfolio")
            st.dataframe(portfolio_df, use_container_width=True, hide_index=True)

    st.download_button(
        "Download Market Data (CSV)",
        data=df.to_csv(index=False).encode("utf-8"),
        file_name="crypto_market_data.csv",
        mime="text/csv",
    )

    st.markdown("---")
    st.subheader("📊 Scraped Market Data")
    st.dataframe(df, use_container_width=True, hide_index=True)

    st.markdown("---")
    st.subheader("📈 Exploratory Data Analysis")
    viz_col1, viz_col2 = st.columns(2)

    with viz_col1:
        fig_mc = px.bar(
            df,
            x="Symbol",
            y="Market Cap ($)",
            color="Market Cap ($)",
            title="Market Capitalization Comparison",
            labels={"Market Cap ($)": "Market Cap (USD)"},
            template="plotly_dark",
        )
        st.plotly_chart(fig_mc, use_container_width=True)

    with viz_col2:
        fig_change = px.bar(
            df,
            x="Symbol",
            y="24h Change (%)",
            color="24h Change (%)",
            color_continuous_scale=["red", "gray", "green"],
            title="24-Hour Price Change (%)",
            template="plotly_dark",
        )
        st.plotly_chart(fig_change, use_container_width=True)
else:
    st.warning("No data retrieved. Click 'Refresh Data' or try again later.")

2026-09-15 23:53:28.604 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 23:53:28.606 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 23:53:28.622 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 23:53:28.630 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 23:53:28.633 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 23:53:28.639 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 23:53:28.641 No runtime found, using MemoryCacheStorageManager
2026-09-15 23:53:28.658 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 23:53:28.660 Thread 'MainThread':